# Deformation map

This notebook will show how to compute a deformation map, i.e. $F_\text{calc}^\text{TAAM} - F_\text{calc}^\text{IAM}$.
We first load the structure information from a .cif file or a .pdb file. Here, a molecule in a $\text{P}2_12_12_1$ box is provided, but this can be replaced with whatever structure you have
Next, we initialize an IAM calculator and a TAAM calculator, set the resolution we want, and compute a deformation map. This is then saved in an .mtz-file.

In [ ]:
import pydiscamb
import io
from cctbx import miller
from iotbx import pdb
from pathlib import Path

# Replace this with a path to your structure file
structure_file = Path().resolve() / "data" / "tyrosine.pdb"

# Load structure
xrs = pdb.input(file_name=str(structure_file)).xray_structure_simple()
xrs.scattering_type_registry(table="xray")

# Prepare calculators
iam_calculator = pydiscamb.DiscambWrapper(
    xrs,
    method=pydiscamb.FCalcMethod.IAM,
)
taam_calculator = pydiscamb.DiscambWrapper(
    xrs,
    method=pydiscamb.FCalcMethod.TAAM,
)

# Log the number of typed atoms
buffer = io.StringIO()
taam_calculator.show_atom_type_assignment(buffer)
log_content = buffer.getvalue()
print(log_content)
with open("deformation_map_atom_typing.log", "w") as f:
    f.write(log_content)

# Set resolution
miller_set = miller.build_set(
    crystal_symmetry=xrs.crystal_symmetry(), 
    anomalous_flag=False, 
    d_min=0.7,
)
iam_calculator.set_indices(miller_set.indices())
taam_calculator.set_indices(miller_set.indices())

# Compute structure factors
fcalc_iam = iam_calculator.f_calc()
fcalc_taam = taam_calculator.f_calc()

# Compute deformation map
deformation_map = fcalc_taam - fcalc_iam


# Create a miller.array to save
deformation_map_array = miller_set.array(
    data=deformation_map,
)
deformation_map_array.write_mtz(
    file_name="deformation_map.mtz",
)

Below is a capture from Coot, showing the resulting deformation map at 3.0 RMSD

![tyrosine_difference_map.png](./data/tyrosine_difference_map.png)


# Even easier
If the goal is simply to compute the deformation map at a given resolution, it can be performed even simpler:

In [ ]:
w1 = pydiscamb.DiscambWrapper.from_file("data/tyrosine.pdb", pydiscamb.FCalcMethod.IAM)
w2 = pydiscamb.DiscambWrapper.from_file("data/tyrosine.pdb", pydiscamb.FCalcMethod.TAAM)

d_min = 1.5
fc1 = w1.f_calc(d_min)
fc2 = w2.f_calc(d_min)

deformation_map = fc2 - fc1